In [1]:
import json
import time
from pathlib import Path

import pandas as pd

from lolalytics_api import matchup


# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

TARGET_PATCH = "16.19"
RANK = "emerald"

PROCESSED_DIR = Path("data/processed")

INPUT_FILE = PROCESSED_DIR / "unique_matchups_16_19.csv"

OUTPUT_CSV = PROCESSED_DIR / "expected_winrates_16_19.csv"
OUTPUT_PARQUET = PROCESSED_DIR / "expected_winrates_16_19.parquet"

In [2]:
unique_matchups = pd.read_csv(INPUT_FILE)

print("Unique matchups to scrape:", len(unique_matchups))

unique_matchups.head(20)

Unique matchups to scrape: 121


,lane,champion,opponent_champion
0,adc,Corki,Jhin
1,adc,Jinx,Yunara
2,adc,Jinx,Zeri
3,adc,Kaisa,Ashe
4,adc,Kaisa,Yunara
5,adc,Lucian,Kaisa
6,adc,Mel,Swain
7,adc,Tristana,Jinx
8,adc,Tristana,Yasuo
9,adc,Yunara,Tristana


In [3]:
def lolalytics_name(champion):
    name = str(champion).lower()

    aliases = {
        "monkeyking": "wukong",
    }

    return aliases.get(name, name)

unique_matchups["lol_champion"] = (
    unique_matchups["champion"]
    .apply(lolalytics_name)
)

unique_matchups["lol_opponent"] = (
    unique_matchups["opponent_champion"]
    .apply(lolalytics_name)
)

unique_matchups["lol_lane"] = (
    unique_matchups["lane"]
    .str.lower()
)

In [4]:
# --------------------------------------------------
# LOAD EXISTING RESULTS IF THEY EXIST
# --------------------------------------------------

if OUTPUT_CSV.exists():
    existing = pd.read_csv(OUTPUT_CSV)

    print(
        f"Loaded {len(existing)} existing "
        f"matchup results"
    )

else:
    existing = pd.DataFrame(
        columns=[
            "patch",
            "rank",
            "lane",
            "champion",
            "opponent_champion",
            "expected_winrate",
            "matchup_games",
            "status"
        ]
    )


# Matchups we've already attempted
completed = set(
    zip(
        existing["lane"],
        existing["champion"],
        existing["opponent_champion"]
    )
)


rows = existing.to_dict("records")

In [5]:
for i, row in unique_matchups.iterrows():

    lane = row["lane"]
    champion = row["champion"]
    opponent = row["opponent_champion"]

    key = (
        lane,
        champion,
        opponent
    )

    # Already scraped
    if key in completed:
        continue


    lol_champion = row["lol_champion"]
    lol_opponent = row["lol_opponent"]
    lol_lane = row["lol_lane"]


    try:

        result = matchup(
            champion1=lol_champion,
            champion2=lol_opponent,
            lane=lol_lane,
            rank=RANK
        )


        # Package sometimes returns JSON as a string
        if isinstance(result, str):
            result = json.loads(result)


        winrate = result.get("winrate")
        games = result.get("number_of_games")


        # ------------------------------------------
        # Convert values
        # ------------------------------------------

        if winrate is not None:
            winrate = float(
                str(winrate)
                .replace("%", "")
                .strip()
            )


        if games is not None:
            games = int(
                str(games)
                .replace(",", "")
                .strip()
            )


        rows.append({

            "patch": TARGET_PATCH,
            "rank": RANK,

            "lane": lane,

            "champion": champion,

            "opponent_champion":
                opponent,

            "expected_winrate":
                winrate,

            "matchup_games":
                games,

            "status": "ok"
        })


        print(
            f"{len(rows)}/{len(unique_matchups)} | "
            f"{lane}: "
            f"{champion} vs {opponent} -> "
            f"{winrate}% ({games} games)"
        )


    except Exception as e:

        rows.append({

            "patch": TARGET_PATCH,
            "rank": RANK,

            "lane": lane,

            "champion": champion,

            "opponent_champion":
                opponent,

            "expected_winrate":
                None,

            "matchup_games":
                None,

            "status":
                f"error: {type(e).__name__}"
        })


        print(
            f"FAILED | {lane}: "
            f"{champion} vs {opponent} | "
            f"{e}"
        )


    # ------------------------------------------
    # SAVE AFTER EVERY REQUEST
    # ------------------------------------------

    expected_winrates = pd.DataFrame(
        rows
    )

    expected_winrates.to_csv(
        OUTPUT_CSV,
        index=False
    )

    expected_winrates.to_parquet(
        OUTPUT_PARQUET,
        index=False
    )


    # Don't hammer LoLalytics
    time.sleep(0.2)

1/121 | adc: Corki vs Jhin -> 46.67% (345 games)
2/121 | adc: Jinx vs Yunara -> 53.1% (4699 games)
3/121 | adc: Jinx vs Zeri -> 50.68% (1178 games)
4/121 | adc: Kaisa vs Ashe -> 51.4% (1862 games)
5/121 | adc: Kaisa vs Yunara -> 47.75% (4775 games)
6/121 | adc: Lucian vs Kaisa -> 49.18% (2621 games)
7/121 | adc: Mel vs Swain -> 52.83% (53 games)
8/121 | adc: Tristana vs Jinx -> 48.35% (3160 games)
9/121 | adc: Tristana vs Yasuo -> 51.38% (1481 games)
10/121 | adc: Yunara vs Tristana -> 50.52% (2482 games)
11/121 | jungle: Chogath vs Nunu -> 52.08% (192 games)
12/121 | jungle: Ekko vs Belveth -> 49.13% (173 games)
13/121 | jungle: Ekko vs Kayn -> 52.11% (616 games)
14/121 | jungle: Ekko vs LeeSin -> 52.67% (748 games)
15/121 | jungle: Ekko vs Naafiri -> 48.35% (182 games)
16/121 | jungle: Elise vs Sylas -> 56.33% (316 games)
17/121 | jungle: Graves vs Briar -> 51.17% (641 games)
18/121 | jungle: Graves vs Lillia -> 50.0% (532 games)
19/121 | jungle: Gwen vs Rammus -> 55.56% (36 games)
2

In [6]:
expected_winrates = pd.read_csv(
    OUTPUT_CSV
)

expected_winrates = (
    expected_winrates
    .drop_duplicates(
        subset=[
            "lane",
            "champion",
            "opponent_champion"
        ],
        keep="last"
    )
    .reset_index(drop=True)
)


expected_winrates.to_csv(
    OUTPUT_CSV,
    index=False
)

expected_winrates.to_parquet(
    OUTPUT_PARQUET,
    index=False
)


print(
    "Total matchup rows:",
    len(expected_winrates)
)

print(
    "Successful:",
    (
        expected_winrates["status"]
        == "ok"
    ).sum()
)

print(
    "Failed:",
    (
        expected_winrates["status"]
        != "ok"
    ).sum()
)

expected_winrates.head(20)

Total matchup rows: 121
Successful: 119
Failed: 2


,patch,rank,lane,champion,opponent_champion,expected_winrate,matchup_games,status
0,16.19,emerald,adc,Corki,Jhin,46.67,345.0,ok
1,16.19,emerald,adc,Jinx,Yunara,53.10,4699.0,ok
2,16.19,emerald,adc,Jinx,Zeri,50.68,1178.0,ok
3,16.19,emerald,adc,Kaisa,Ashe,51.40,1862.0,ok
4,16.19,emerald,adc,Kaisa,Yunara,47.75,4775.0,ok
5,16.19,emerald,adc,Lucian,Kaisa,49.18,2621.0,ok
6,16.19,emerald,adc,Mel,Swain,52.83,53.0,ok
7,16.19,emerald,adc,Tristana,Jinx,48.35,3160.0,ok
8,16.19,emerald,adc,Tristana,Yasuo,51.38,1481.0,ok
9,16.19,emerald,adc,Yunara,Tristana,50.52,2482.0,ok


In [7]:
failed = expected_winrates[
    expected_winrates["status"] != "ok"
]

failed

,patch,rank,lane,champion,opponent_champion,expected_winrate,matchup_games,status
106,16.19,emerald,support,Udyr,Maokai,NaN,NaN,error: IndexError
114,16.19,emerald,top,Taliyah,Nasus,NaN,NaN,error: IndexError


In [9]:
import pandas as pd

otp_matches = pd.read_parquet(
    "data/processed/otp_matches_16_19.parquet"
)

print(otp_matches.shape)
otp_matches.head()

(129, 15)


,match_id,puuid,seed_tier,patch,side,lane,champion,opponent_champion,opponent_puuid,win,game_duration,game_start_timestamp,champion_games,player_games,champion_share
0,NA1_5647510258,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Vladimir,5dwOvvWd4lI2z8arXUql3b7gyhv-fCFfMSU5xbA9fJ_9rD...,False,2719,1790169107441,9,10,0.900000
1,NA1_5647497734,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Anivia,J3ZfdIi9Q8h5xKzPAlYvOhAOM1OpE85aM1Rv6M-leAsHK3...,False,943,1790172252893,9,10,0.900000
2,NA1_5647517399,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,red,jungle,Nunu,FiddleSticks,qIRl10tZIWyJcyrGwScd1YJ9vXekf_ubDWt7bO054EvF9W...,False,2309,1790175593060,17,19,0.894737
3,NA1_5647521561,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Vex,Ts1tliuj-uuhtj4zLUkYRkjRSuVoFR5zAg5sAV2dfLmGWT...,False,1762,1790176465592,9,10,0.900000
4,NA1_5647532032,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,red,mid,Lucian,Brand,rj980nimbBPg1vHHy3vjP-ocQrXbx8YTDtJf-yMIEWL-SB...,True,1615,1790178290842,1,19,0.052632


In [10]:
otp_matches.merge(
    expected_winrates,
    on=[
        "lane",
        "champion",
        "opponent_champion"
    ],
    how="left"
)

,match_id,puuid,seed_tier,patch_x,side,lane,champion,opponent_champion,opponent_puuid,win,game_duration,game_start_timestamp,champion_games,player_games,champion_share,patch_y,rank,expected_winrate,matchup_games,status
0,NA1_5647510258,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Vladimir,5dwOvvWd4lI2z8arXUql3b7gyhv-fCFfMSU5xbA9fJ_9rD...,False,2719,1790169107441,9,10,0.900000,16.19,emerald,52.31,65.0,ok
1,NA1_5647497734,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Anivia,J3ZfdIi9Q8h5xKzPAlYvOhAOM1OpE85aM1Rv6M-leAsHK3...,False,943,1790172252893,9,10,0.900000,16.19,emerald,38.89,36.0,ok
2,NA1_5647517399,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,red,jungle,Nunu,FiddleSticks,qIRl10tZIWyJcyrGwScd1YJ9vXekf_ubDWt7bO054EvF9W...,False,2309,1790175593060,17,19,0.894737,16.19,emerald,51.79,56.0,ok
3,NA1_5647521561,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Vex,Ts1tliuj-uuhtj4zLUkYRkjRSuVoFR5zAg5sAV2dfLmGWT...,False,1762,1790176465592,9,10,0.900000,16.19,emerald,53.49,43.0,ok
4,NA1_5647532032,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,red,mid,Lucian,Brand,rj980nimbBPg1vHHy3vjP-ocQrXbx8YTDtJf-yMIEWL-SB...,True,1615,1790178290842,1,19,0.052632,16.19,emerald,52.38,21.0,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,NA1_5649068135,-wL2JE9u3WJvgBRtMHRi5q_0gHk497NAD4WQr2M7XMFDOq...,master,16.19,blue,top,Udyr,Tryndamere,rEe_JVyqdmlGT-AdvWglR4AyAwJLN3zv7l1ozet5X9H8ZD...,False,1717,1790394199779,6,6,1.000000,16.19,emerald,51.28,39.0,ok
125,NA1_5649100794,-wL2JE9u3WJvgBRtMHRi5q_0gHk497NAD4WQr2M7XMFDOq...,master,16.19,red,top,Udyr,Kennen,erwL7hsr8nx4zbLrWdesNlKeLWf4GqnKn3tuGC3as3QCC8...,False,1304,1790396211539,6,6,1.000000,16.19,emerald,46.67,15.0,ok
126,NA1_5649106415,CuKY7bDcvLbo_3kmUhWma7EYGZ22lpUecDjzCf4IPnZ-xz...,master,16.19,blue,mid,Zed,Syndra,jFSU8TXal_sBzGiz8V5LGlKaNyl9G5u3jWbsVE9Sr0K0dL...,True,1670,1790397439328,1,1,1.000000,16.19,emerald,51.24,804.0,ok
127,NA1_5649106415,jFSU8TXal_sBzGiz8V5LGlKaNyl9G5u3jWbsVE9Sr0K0dL...,master,16.19,red,mid,Syndra,Zed,CuKY7bDcvLbo_3kmUhWma7EYGZ22lpUecDjzCf4IPnZ-xz...,False,1670,1790397439328,2,20,0.100000,16.19,emerald,50.12,818.0,ok


In [11]:
merged = otp_matches.merge(
    expected_winrates,
    on=[
        "lane",
        "champion",
        "opponent_champion"
    ],
    how="left"
)

merged.head()

,match_id,puuid,seed_tier,patch_x,side,lane,champion,opponent_champion,opponent_puuid,win,game_duration,game_start_timestamp,champion_games,player_games,champion_share,patch_y,rank,expected_winrate,matchup_games,status
0,NA1_5647510258,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Vladimir,5dwOvvWd4lI2z8arXUql3b7gyhv-fCFfMSU5xbA9fJ_9rD...,False,2719,1790169107441,9,10,0.900000,16.19,emerald,52.31,65.0,ok
1,NA1_5647497734,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Anivia,J3ZfdIi9Q8h5xKzPAlYvOhAOM1OpE85aM1Rv6M-leAsHK3...,False,943,1790172252893,9,10,0.900000,16.19,emerald,38.89,36.0,ok
2,NA1_5647517399,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,red,jungle,Nunu,FiddleSticks,qIRl10tZIWyJcyrGwScd1YJ9vXekf_ubDWt7bO054EvF9W...,False,2309,1790175593060,17,19,0.894737,16.19,emerald,51.79,56.0,ok
3,NA1_5647521561,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Vex,Ts1tliuj-uuhtj4zLUkYRkjRSuVoFR5zAg5sAV2dfLmGWT...,False,1762,1790176465592,9,10,0.900000,16.19,emerald,53.49,43.0,ok
4,NA1_5647532032,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,red,mid,Lucian,Brand,rj980nimbBPg1vHHy3vjP-ocQrXbx8YTDtJf-yMIEWL-SB...,True,1615,1790178290842,1,19,0.052632,16.19,emerald,52.38,21.0,ok


In [ ]:
analysis_df = merged.copy()

# Clean duplicate columns created by the LoLalytics merge
if "patch_x" in analysis_df.columns:
    analysis_df = analysis_df.rename(columns={"patch_x": "patch"})

analysis_df = analysis_df.drop(
    columns=["patch_y"],
    errors="ignore"
)

analysis_df = analysis_df.rename(
    columns={"rank": "matchup_rank"}
)

analysis_df["win"] = analysis_df["win"].astype(int)

print(analysis_df.shape)

analysis_df.head()

In [ ]:
import os
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv

from otp_match_pipeline import (
    riot_get,
    download_missing_matches,
    patch_from_match,
    ROUTING,
    TARGET_QUEUE
)


load_dotenv()

RIOT_API_KEY = os.getenv("RIOT_API_KEY")

HEADERS = {
    "X-Riot-Token": RIOT_API_KEY
}


PROCESSED_DIR = Path("data/processed")
RAW_MATCH_DIR = Path("data/otp_raw_matches")

OPPONENT_INDEX_CSV = (
    PROCESSED_DIR / "opponent_match_index.csv"
)

OPPONENT_INDEX_PARQUET = (
    PROCESSED_DIR / "opponent_match_index.parquet"
)


MATCHES_PER_OPPONENT = 100

In [ ]:
opponent_puuids = (
    analysis_df["opponent_puuid"]
    .dropna()
    .unique()
)


# ---------------------------------------
# Load existing opponent history index
# ---------------------------------------

if OPPONENT_INDEX_CSV.exists():

    opponent_index = pd.read_csv(
        OPPONENT_INDEX_CSV
    )

else:

    opponent_index = pd.DataFrame(
        columns=[
            "opponent_puuid",
            "match_id"
        ]
    )


print(
    "Unique opponents needed:",
    len(opponent_puuids)
)

print(
    "Opponents already indexed:",
    opponent_index["opponent_puuid"]
    .nunique()
    if len(opponent_index)
    else 0
)

In [ ]:
PLAYER_INDEX = (
    PROCESSED_DIR / "player_match_index.parquet"
)

if PLAYER_INDEX.exists():

    player_index = pd.read_parquet(
        PLAYER_INDEX
    )

    reusable = (
        player_index[
            player_index["puuid"]
            .isin(opponent_puuids)
        ][
            ["puuid", "match_id"]
        ]
        .rename(
            columns={
                "puuid":
                "opponent_puuid"
            }
        )
    )

    opponent_index = pd.concat(
        [
            opponent_index,
            reusable
        ],
        ignore_index=True
    )

    opponent_index = (
        opponent_index
        .drop_duplicates(
            [
                "opponent_puuid",
                "match_id"
            ]
        )
        .reset_index(drop=True)
    )

In [ ]:
already_indexed = set(
    opponent_index[
        "opponent_puuid"
    ].unique()
)

new_opponents = [
    puuid
    for puuid in opponent_puuids
    if puuid not in already_indexed
]


print(
    "New opponent histories to fetch:",
    len(new_opponents)
)


new_rows = []


for i, puuid in enumerate(
    new_opponents,
    start=1
):

    url = (
        f"https://{ROUTING}.api.riotgames.com"
        f"/lol/match/v5/matches/by-puuid/"
        f"{puuid}/ids"
    )

    params = {
        "queue": TARGET_QUEUE,
        "start": 0,
        "count": MATCHES_PER_OPPONENT
    }

    try:

        match_ids = riot_get(
            url,
            HEADERS,
            params
        )

    except Exception as e:

        print(
            f"Skipped opponent "
            f"{i}/{len(new_opponents)}: {e}"
        )

        continue


    for match_id in match_ids:

        new_rows.append({
            "opponent_puuid": puuid,
            "match_id": match_id
        })


    print(
        f"Opponent "
        f"{i}/{len(new_opponents)}: "
        f"{len(match_ids)} matches"
    )

In [ ]:
if new_rows:

    opponent_index = pd.concat(
        [
            opponent_index,
            pd.DataFrame(new_rows)
        ],
        ignore_index=True
    )


opponent_index = (
    opponent_index
    .drop_duplicates(
        [
            "opponent_puuid",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


opponent_index.to_csv(
    OPPONENT_INDEX_CSV,
    index=False
)

opponent_index.to_parquet(
    OPPONENT_INDEX_PARQUET,
    index=False
)


print(
    "Opponent-match rows:",
    len(opponent_index)
)

In [ ]:
opponent_download_index = (
    opponent_index
    .rename(
        columns={
            "opponent_puuid": "puuid"
        }
    )
    .copy()
)

opponent_download_index[
    "seed_tier"
] = "opponent"

opponent_download_index = (
    opponent_download_index[
        [
            "puuid",
            "match_id",
            "seed_tier"
        ]
    ]
)


MAX_NEW_OPPONENT_DOWNLOADS = 2000


cached, remaining, downloaded = (
    download_missing_matches(
        opponent_download_index,
        RAW_MATCH_DIR,
        HEADERS,
        MAX_NEW_OPPONENT_DOWNLOADS
    )
)


print()
print("Cached before:", cached)
print("Missing before:", remaining)
print("Downloaded now:", downloaded)

In [ ]:
import json


TARGET_PATCH = "16.19"

opponent_usage_rows = []


for row in opponent_index.itertuples(
    index=False
):

    file_path = (
        RAW_MATCH_DIR
        / f"{row.match_id}.json"
    )

    if not file_path.exists():
        continue


    try:

        with file_path.open(
            "r",
            encoding="utf-8"
        ) as f:

            match_data = json.load(f)


        if (
            patch_from_match(match_data)
            != TARGET_PATCH
        ):
            continue


        participants = (
            match_data["info"]
            ["participants"]
        )


        player = next(
            (
                p for p in participants
                if p.get("puuid")
                == row.opponent_puuid
            ),
            None
        )


        if player is None:
            continue


        champion = player.get(
            "championName"
        )


        if not champion:
            continue


        opponent_usage_rows.append({
            "opponent_puuid":
                row.opponent_puuid,

            "match_id":
                row.match_id,

            "opponent_champion":
                champion
        })


    except (
        OSError,
        ValueError,
        KeyError,
        TypeError
    ):
        continue

In [ ]:
opponent_games = pd.DataFrame(
    opponent_usage_rows
)

opponent_games = (
    opponent_games
    .drop_duplicates(
        [
            "opponent_puuid",
            "match_id"
        ]
    )
)

print(
    "Usable opponent 16.19 games:",
    len(opponent_games)
)

opponent_games.head()

In [ ]:
opponent_player_counts = (
    opponent_games
    .groupby(
        "opponent_puuid"
    )
    .size()
    .rename(
        "opponent_player_games"
    )
    .reset_index()
)


opponent_champion_counts = (
    opponent_games
    .groupby(
        [
            "opponent_puuid",
            "opponent_champion"
        ]
    )
    .size()
    .rename(
        "opponent_champion_games"
    )
    .reset_index()
)

opponent_specialization = (
    opponent_champion_counts
    .merge(
        opponent_player_counts,
        on="opponent_puuid",
        how="left"
    )
)


opponent_specialization[
    "opponent_champion_share"
] = (
    opponent_specialization[
        "opponent_champion_games"
    ]
    /
    opponent_specialization[
        "opponent_player_games"
    ]
)


opponent_specialization.head(20)

In [ ]:
analysis_df = analysis_df.merge(
    opponent_specialization,
    on=[
        "opponent_puuid",
        "opponent_champion"
    ],
    how="left"
)

In [ ]:
analysis_df[
    [
        "champion",
        "opponent_champion",

        "champion_share",
        "opponent_champion_share",

        "expected_winrate",
        "matchup_games",

        "win"
    ]
].head(20)

In [ ]:
print(
    "Total analysis rows:",
    len(analysis_df)
)

print(
    "Opponent share available:",
    analysis_df[
        "opponent_champion_share"
    ].notna().sum()
)

print(
    "Opponent share missing:",
    analysis_df[
        "opponent_champion_share"
    ].isna().sum()
)

In [ ]:
analysis_df[
    analysis_df[
        "opponent_champion_share"
    ].isna()
][
    [
        "opponent_puuid",
        "opponent_champion"
    ]
].drop_duplicates().head(20)

In [ ]:
analysis_df.to_csv(
    "data/processed/"
    "otp_analysis_16_19.csv",
    index=False
)

analysis_df.to_parquet(
    "data/processed/"
    "otp_analysis_16_19.parquet",
    index=False
)